In [ ]:
# TODO
# Wrap find, filter, mm inside a function
# Wrap reading from find, filter, mm inside a function too

In [1]:
import pandas as pd
import numpy as np
from Bio import SeqIO
from collections import Counter

import generate_spacers
import re
import bioframe as bf
import bioframe.vis


import matplotlib.pyplot as plt
import subprocess
%matplotlib inline

In [2]:
headpath = '/home/gutil/scratch/2023-11-15_Mismatch-CRISPRi-Essential-Genes/'
SEED_LENGTH = 9
SPACER_LENGTH = 20

BIN_DICT = {
    0:[-0.5, 0.1], # Lower bound 
    1:[0.1, 0.2],
    2:[0.2, 0.3],
    3:[0.3, 0.4],
    4:[0.4, 0.5],
    5:[0.5, 0.6],
    6:[0.6, 0.7],
    7:[0.7, 0.8],
    8:[0.8, 0.9],
    9:[0.9, 1.5] # Cannot set upper bound bc there are fitness values >= 1
}

path_to_script = '/home/gutil/lag_paulsson-lab/library_design/generate_sgrnas.py'
path_to_params = headpath+ 'model_param_bsub.csv'
# path_to_genbank = headpath+'basu168_BSGAtlas_modified.gb'
path_to_genbank = headpath+'PY79_BSGAtlas_RATT_manually_curated_modified.gb'
path_log = headpath+'log.txt'

# Biofilm-inducing knock-downs
genes_to_remove = ['sinR', 'abrB', 'ywcC'] # slrC is ywcC
# Add suffix
suffix_to_remove = 'CCAACGGTATCGATAGAACGC' # oLAG322
suffix_other = 'CATCAGGATCCGAATCGTGTC' # oLAG323

rec= SeqIO.read(path_to_genbank, "genbank")
upper_seq = str(rec.seq)
lower_seq = str(rec.seq.reverse_complement())
GENOME_SIZE = len(upper_seq)

headpath = '/home/gutil/scratch/2023-11-15_Mismatch-CRISPRi-Essential-Genes/'
filename_essentials = 'Table_S3_1-s2.0-S2405471220303665-mmc4.xlsx'
filename_controls = 'Table_S3_1-s2.0-S2405471220303665-mmc4.xlsx'
filename_auxotrophs = 'KO_S4_mmc5.xlsx'

In [ ]:
## Change locus tags commas to semicolons

# input_gb = headpath+'PY79_BSGAtlas_RATT_manually_curated.gb'
# output_gb = headpath+'PY79_BSGAtlas_RATT_manually_curated_modified.gb' # With semicolon

# generate_spacers.change_commas_to_semicolon_in_locus_tag(input_gb, output_gb)

# Generate dataframe of genes

In [ ]:
df_genes = generate_spacers.load_genes(path_to_genbank)
df_genes = generate_spacers.classify_genes(df_genes, headpath, filename_essentials, filename_auxotrophs)

In [3]:
# df_genes.to_pickle(headpath+'df_genes.pkl')
df_genes = pd.read_pickle(headpath+'df_genes.pkl')

# Essential genes

In [ ]:
df = (pd.read_excel(headpath + filename_essentials,
                    sheet_name="relative fitness (bsu data)")
    .loc[lambda df_: df_["family_retained"] == True] # Remove aberrant gRNAs according to the paper
    .drop(columns=["family_retained"]) # Drop column
    .rename(columns={'relative fitness (predicted)':'predicted_efficacy',
                     'relative fitness (mean)':'relative_fitness_mean',
                     'relative fitness (stddev)':'relative_fitness_std',
                     'variant': 'mismatch_sgrna',
                     'original': 'full_sgrna'}) # Rename column
    .astype({"pam": "category"}) # Set type of pam to categorical
    .dropna(subset=['relative_fitness_mean'])
    .assign(locus_tag = lambda df_: df_['locus_tag'].str.slice(0, 3)+'_'+df_['locus_tag'].str.slice(3)) # Add underscore to locus tag
    .sort_values(["locus_tag", "relative_fitness_mean"])
    # Assign the gene names from the reference
    .rename(columns={'gene': 'gene_name_data'})
    .merge(df_genes.loc[lambda df_:df_['category_gene']=='essential_from_dataset', ['locus_tag', 'gene']],
           on='locus_tag',
           how='left')

    # Assign bin indices
    .assign(bin_index_fitness = lambda df_:generate_spacers.get_bin_ids(df_,
                                                                    col='relative_fitness_mean',
                                                                    bin_dict=BIN_DICT))
    .loc[lambda df_: df_["bin_index_fitness"] != -1] # Remove rows with bin_index_fitness == -1
    .assign(bin_index = lambda df_:generate_spacers.get_bin_ids(df_,
                                                               col='predicted_efficacy',
                                                               bin_dict=BIN_DICT))
    .loc[lambda df_: df_["bin_index"] != -1] # Remove rows with bin_index == -1

    .loc[lambda df_: df_['mismatch_sgrna'].duplicated(keep=False) == False] # Remove duplicate sgRNAs (both of them)   
    # Select the rows whose mismatch_grna does not contain any restriction site
    .loc[lambda df_: df_['mismatch_sgrna'].apply(generate_spacers.check_all_sites) == False]

    # Select the rows whose sequence to be ordered does not contain BsaI or PvUI restriction sites
    .loc[lambda df_: df_['mismatch_sgrna'].apply(generate_spacers.add_adapter_seqs).apply(generate_spacers.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False]
    .reset_index(drop=True)

    # This is to use df_hits to get the off-targets
    .rename(columns={'mismatch_sgrna': 'query_spacer'})
    .assign(query_end_position = None)

    .assign(#gc_downstream_8_9 = lambda df_: df_['downstream'].str[7:9].str.count('G') + df_['downstream'].str[7:9].str.count('C'),
         offset_gt_1000 = lambda df_: df_['offset'] > 1000,
          # Assign a column that counts the number of Ts in the first 3 bases of the query_spacer
         n_t_in_start = lambda df_: df_['query_spacer'].str[:3].str.count('T'),
         agg_pam = lambda df_: df_['pam'].str.contains('AGG'))
    # Get seed sequence to find hits
    .assign(seed = lambda df_: df_['query_spacer'].str.slice(20 - SEED_LENGTH, 20))
)

In [ ]:
# Get unique seed sequences
df_seeds = (df
            .loc[:,['seed']]
            .drop_duplicates()
            .reset_index(drop=True)
)

df_seeds_all = generate_spacers.find_all_hits_seeds_parallel(df_seeds, upper_seq, lower_seq)
# df_seeds_all = generate_spacers.find_all_hits_seeds(df_seeds, upper_seq, lower_seq)
df_seeds_all.to_pickle(headpath+'df_seeds_all_essentials_from_dataset.pkl')
# df_seeds_all = pd.read_pickle(headpath+'df_seeds_all_essentials_from_dataset.pkl')

In [ ]:
df_hits = (df
           .merge(df_seeds_all, on='seed', how='left')
           .assign(n_nts_mismatched = lambda df_: df_.apply(lambda row: generate_spacers.count_differences(row['full_sgrna'], row['spacer_seq']), axis=1))
           .loc[lambda df_: ~df_['n_nts_mismatched'].isna()] # Keep only off-targets (remove the ones with mismatches on the seed)
           .loc[lambda df_: df_['n_nts_mismatched'] > 0] # Keep only off-targets
           .astype({"start": 'int64', "end": 'int64'})
)

df = (df
 .join(df_hits.groupby('query_spacer').size().rename('n_off_targets'), on='query_spacer', how='left')
 .fillna({'n_off_targets': 0})
 .astype({'n_off_targets': 'uint32'})
)

In [ ]:
df_hits_gene_worst = (bioframe.overlap(df1 = df_hits,
                  df2 = df_genes.assign(chrom = '0').loc[:, ['locus_tag', 'gene', 'matches_up_strand', 'start', 'end', 'chrom']],
                  suffixes=('', '_off_target'),
                  how = 'left')
        .drop(columns = ['chrom_off_target', 'chrom'])
        # .astype({'matches_up_strand_gene': 'boolean'})
        # Binds non-template (coding) strand is equivalent to the spacer sequence matching the template (non-coding) strand of the gene
        .assign(binds_non_template_strand_off_target = lambda df_: df_['matches_up_strand'] != df_['matches_up_strand_off_target'])
        # .loc[lambda df_: df_['query_end_position_seed'] != df_['seed_end_seed']] # Filter out the on-targets
        .sort_values(by = ['locus_tag', 'query_spacer', 'binds_non_template_strand_off_target', 'n_nts_mismatched'],
                     ascending = [True, True, False, True])
        # Remove query_spacers whose full_sgrna has any off target to a gene in genes_to_remove (biofilm-inhibiting genes)
        # .loc[lambda df_:~df_['full_sgrna'].isin(
        #     df_.loc[df_['gene_off_target'].isin(genes_to_remove), 'full_sgrna'].unique())]
        .reset_index(drop=True)
)

In [ ]:
# Get full_sgrnas to remove
spacers_to_remove = df_hits_gene_worst.loc[df_hits_gene_worst['gene_off_target'].isin(genes_to_remove), 'full_sgrna'].unique()

df_hits_gene_worst = (df_hits_gene_worst
                  .groupby('query_spacer')
                  .head(1)
)

In [ ]:
# MERGE THIS BACK WITH THE ORIGINAL DATAFRAME TO HAVE IT READY FOR FILTERING/BINNING
cols_first = ['locus_tag', 'gene', 'bin_index_fitness', 'bin_index', 'n_off_targets', 'n_nts_mismatched', 'binds_non_template_strand_off_target']
df = (df
        # Remove spacers that will off-target to biofilm-reppressing genes
        .loc[lambda df_:~df_['full_sgrna'].isin(spacers_to_remove)]
        
        .merge(df_hits_gene_worst.drop(columns = set(df.columns) & set(df_hits_gene_worst.columns) - set(['query_spacer'])), on='query_spacer', how='left')
        .reset_index(drop=True)
        .loc[:, lambda df_: cols_first + [col for col in df_.columns if col not in cols_first]]
        .assign(selected = False)
        # Do a hard filter: Include only guides with less than 10 off-targets
        .loc[lambda df_: df_['n_off_targets'] <= 10]
        # .columns
)

In [ ]:
sort_by = ['n_off_targets', 'binds_non_template_strand_off_target', 'n_nts_mismatched',
                     'agg_pam', 'n_t_in_start', 'offset_gt_1000']
sort_ascending = [True, True, False,
                            True, True, True]

df = (df
    # Assign the bin_index_fitness and bin_index (predicted)
    .pipe(generate_spacers.subsample_spacers_per_bin,
          k = 1,
          sort_by = sort_by,
          sort_ascending = sort_ascending,
          bin_index_name = 'bin_index_fitness')
    .pipe(generate_spacers.subsample_spacers_per_bin,
          k = 1,
          sort_by = sort_by,
          sort_ascending = sort_ascending,
          bin_index_name = 'bin_index')

    # Assign the missing bin_index_fitness by distance to the closest bin_index_fitness
    .groupby(['locus_tag'])
    .apply(generate_spacers.generate_bin_sample_single_gene,
           bin_dict = BIN_DICT,
           k = 1,
           sort_by = sort_by,
           sort_ascending = sort_ascending,
           bin_index_name = 'bin_index_fitness',
           include_groups = False)
    .reset_index()
    .drop(columns = 'level_1')

    # Assign the missing bin_index by distance to the closest bin_index
    .groupby(['locus_tag'])
    .apply(generate_spacers.generate_bin_sample_single_gene,
           bin_dict = BIN_DICT,
           k = 1,
           sort_by = sort_by,
           sort_ascending = sort_ascending,
           bin_index_name = 'bin_index',
           include_groups = False)
    .reset_index()
    .drop(columns = 'level_1')

    .sort_values(['locus_tag', 'bin_index']+sort_by, ascending=[True, True]+sort_ascending)
#     .loc[lambda df_: df_['selected']]

)

In [ ]:
df_selected = df.loc[lambda df_: df_['selected']]
df_selected.to_pickle(headpath+'df_selected_essentials_from_dataset.pkl')
df.to_pickle(headpath+'df_essentials_from_dataset.pkl')

In [ ]:
import matplotlib.pyplot as plt

gene = 'dnaX'
plt.scatter(df[df['gene']==gene]['predicted_efficacy'], df[df['gene']==gene]['relative_fitness_mean'])#, c=df[df['gene']==gene]['n_off_targets'], cmap='viridis', s=100)
plt.scatter(df_selected[df_selected['gene']==gene]['predicted_efficacy'], df_selected[df_selected['gene']==gene]['relative_fitness_mean'], c='red', s=100)
plt.ylim([-.3, 1.3])
plt.show()


In [ ]:
df['n_off_targets'].hist(density=True, alpha=0.5, label='All')
df_selected['n_off_targets'].hist(density=True, alpha=0.5, label='Selected')
plt.xlabel('Number of off-targets per gRNA', fontsize=20)
# Remove grid
plt.grid(False)
# Set background to black and text to white
plt.gca().set_facecolor('black')
# Set also outer frame to black
plt.box(False)
plt.legend(shadow=False, fontsize=20)
# Make tick marks bigger
plt.tick_params(axis='both', which='major', labelsize=20)
plt.show()

In [ ]:
# Get the fraction of gRNAs with 0 off targets per locus_tag
(df
 .groupby('locus_tag')
 .apply(lambda df_: (df_['n_off_targets'] == 0).sum() / len(df_), include_groups=False)
 .hist(density = True, alpha = 0.5, label='All'))
# Same for df_selected
(df_selected
 .groupby('locus_tag')
 .apply(lambda df_: (df_['n_off_targets'] == 0).sum() / len(df_), include_groups=False)
 .hist(density = True, alpha = 0.5, label='Selected')
)

plt.xlabel('Fraction spacers with\nzero off-targets per gene', fontsize=20)
# Remove grid
plt.grid(False)
# Set background to black and text to white
plt.gca().set_facecolor('black')
# Set also outer frame to black
plt.box(False)
plt.legend(shadow=False, fontsize=20)
# Make tick marks bigger
plt.tick_params(axis='both', which='major', labelsize=20)
plt.show()

# Controls

In [ ]:
# Do it for all the controls
# Read Excel file
# table_url = "https://ars.els-cdn.com/content/image/1-s2.0-S2405471220303665-mmc4.xlsx"
filename_controls = 'Table_S3_1-s2.0-S2405471220303665-mmc4.xlsx'
df = (pd.read_excel(headpath+filename_controls,
                    sheet_name="relative fitness (bsu controls)")
                 # Remove row with nans
                 .rename(columns={'relative fitness (stddev)':'relative_fitness_std',
                                 'relative fitness (mean)':'relative_fitness_mean',
                                 'variant': 'mismatch_sgrna',
                                 'original': 'full_sgrna'}) # Rename column
                 .dropna(subset=['relative_fitness_mean'])
                 .loc[lambda df_: df_['mismatch_sgrna'].apply(generate_spacers.check_all_sites) == False] # Remove rows with restriction sites
                 .loc[lambda df_: df_['mismatch_sgrna'].apply(generate_spacers.add_adapter_seqs).apply(generate_spacers.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Remove rows with BsaI and PvuI sites
                 # Remove rows for which 'relative fitness (stdev)' is less than 0.02
                #  .loc[lambda df_: df_['relative_fitness_std'] < 0.02]
                 # Filter rows whose relative_fitness_mean is between 1 + delta and 1 - delta
                 .loc[lambda df_: df_['relative_fitness_mean'].between(1-0.05, 1+0.5)] # TODO: Let delta be variable?
               #   .sample(n=100, replace=False) # Sample 100 controls
                 .assign(seed = lambda df_: df_['full_sgrna'].str[-9:],
                         category = 'control')
                 .astype({'category': 'category'})
                #  .assign(seq_to_order = lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs))
                #  .loc[lambda df_: df_['seq_to_order'].apply(check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Check for BsaI and PvuI sites only
                 .reset_index(drop=True)
                 .rename(columns={'mismatch_sgrna': 'query_spacer'})
                 .assign(query_end_position = None)
    )

In [ ]:
# Get unique seed sequences
df_seeds = (df
            .loc[:,['seed']]
            .drop_duplicates()
            .reset_index(drop=True)
)

df_seeds_all = generate_spacers.find_all_hits_seeds_parallel(df_seeds, upper_seq, lower_seq)
# df_seeds_all = generate_spacers.find_all_hits_seeds(df_seeds, upper_seq, lower_seq)
df_seeds_all.to_pickle(headpath+'df_seeds_all_controls.pkl')
# df_seeds_all = pd.read_pickle(headpath+'df_seeds_all_controls.pkl')

In [ ]:
df_hits = (df
           .merge(df_seeds_all, on='seed', how='left')
           .assign(n_nts_mismatched = lambda df_: df_.apply(lambda row: generate_spacers.count_differences(row['full_sgrna'], row['spacer_seq']), axis=1))
           .loc[lambda df_: ~df_['n_nts_mismatched'].isna()] # Keep only off-targets (remove the ones with mismatches on the seed)
        #    .loc[lambda df_: df_['n_nts_mismatched'] > 0] # Keep only off-targets
           .astype({"start": 'int64', "end": 'int64'})
)           

df = (df
 .join(df_hits.groupby('query_spacer').size().rename('n_off_targets'), on='query_spacer', how='left')
 .fillna({'n_off_targets': 0})
 .astype({'n_off_targets': 'uint32'})
)

In [ ]:
df_hits_gene_worst = (bioframe.overlap(df1 = df_hits,
                  df2 = df_genes.assign(chrom = '0').loc[:, ['locus_tag', 'gene', 'matches_up_strand', 'start', 'end', 'chrom']],
                  suffixes=('', '_off_target'),
                  how = 'left')
        .drop(columns = ['chrom_off_target', 'chrom'])
        # .astype({'matches_up_strand_gene': 'boolean'})
        # Binds non-template (coding) strand is equivalent to the spacer sequence matching the template (non-coding) strand of the gene
        .assign(binds_non_template_strand_off_target = lambda df_: df_['matches_up_strand'] != df_['matches_up_strand_off_target'])
        # .loc[lambda df_: df_['query_end_position_seed'] != df_['seed_end_seed']] # Filter out the on-targets
        .sort_values(by = ['query_spacer', 'binds_non_template_strand_off_target', 'n_nts_mismatched'],
                     ascending = [True, False, True])
        # Remove query_spacers whose full_sgrna has any off target to a gene in genes_to_remove (biofilm-inhibiting genes)
        # .loc[lambda df_:~df_['full_sgrna'].isin(
        #     df_.loc[df_['gene_off_target'].isin(genes_to_remove), 'full_sgrna'].unique())]
        .reset_index(drop=True)
)

In [ ]:
# Get full_sgrnas to remove
spacers_to_remove = df_hits_gene_worst.loc[df_hits_gene_worst['gene_off_target'].isin(genes_to_remove), 'full_sgrna'].unique()

df_hits_gene_worst = (df_hits_gene_worst
                  .groupby('query_spacer')
                  .head(1)
)

In [ ]:
# MERGE THIS BACK WITH THE ORIGINAL DATAFRAME TO HAVE IT READY FOR FILTERING/BINNING
cols_first = ['n_off_targets', 'binds_non_template_strand_off_target', 'n_nts_mismatched']
df = (df
        # Remove spacers that will off-target to biofilm-reppressing genes
        .loc[lambda df_:~df_['full_sgrna'].isin(spacers_to_remove)]
        
        .merge(df_hits_gene_worst.drop(columns = set(df.columns) & set(df_hits_gene_worst.columns) - set(['query_spacer'])), on='query_spacer', how='left')
        .reset_index(drop=True)
        .loc[:, lambda df_: cols_first + [col for col in df_.columns if col not in cols_first]]
        .assign(selected = False)
        # Do a hard filter: Include only guides with less than 10 off-targets
        .loc[lambda df_: df_['n_off_targets'] <= 10]
        # Sort for the controls
        .sort_values(by = ['n_off_targets', 'binds_non_template_strand_off_target', 'n_nts_mismatched'], ascending = [True, True, False])
        # .columns
)

In [ ]:
N_CONTROLS = 500
df_selected = df.head(N_CONTROLS)

df.to_pickle(headpath+'df_controls_from_dataset.pkl')
df_selected.to_pickle(headpath + 'df_selected_controls_from_dataset.pkl')

# Auxotrophs


In [ ]:
df_genes = pd.read_pickle(headpath+'df_genes.pkl')

In [ ]:
path_to_output = headpath+'sgrnas_auxotrophs'
locus_tags_auxotrophs = df_genes.loc[df_genes['category_gene']=='auxotroph','locus_tag_file'].to_list()

generate_spacers.run_grna_finding_pipeline(locus_tags_auxotrophs,
                                            path_to_script,
                                            path_to_genbank,
                                            path_to_params,
                                            path_log,
                                            path_to_output,
                                            get_mismatches = True)

In [ ]:
df = generate_spacers.load_and_process_dfs_from_pipeline(path_to_output, df_genes, BIN_DICT)

In [ ]:
# Get unique seed sequences
df_seeds = (df
            .loc[:,['seed']]
            .drop_duplicates()
            .reset_index(drop=True)
)

df_seeds_all = generate_spacers.find_all_hits_seeds_parallel(df_seeds, upper_seq, lower_seq)
# df_seeds_all = generate_spacers.find_all_hits_seeds(df_seeds, upper_seq, lower_seq)
# df_seeds_all.to_pickle(headpath+'df_seeds_all_auxotrophs.pkl')
# df_seeds_all = pd.read_pickle(headpath+'df_seeds_all_auxotrophs.pkl')

In [ ]:
# Get information about the off_targets per each query_spacer
df_hits = (df
           .merge(df_seeds_all, on='seed', how='left')
           .assign(n_nts_mismatched = lambda df_: df_.apply(lambda row: generate_spacers.count_differences(row['full_sgrna'], row['spacer_seq']), axis=1))
           .loc[lambda df_: ~df_['n_nts_mismatched'].isna()] # Keep only off-targets (remove the ones with mismatches on the seed)
           .loc[lambda df_: df_['n_nts_mismatched'] > 0] # Keep only off-targets
           .astype({"start": 'int64', "end": 'int64'})
) 

# Add the number of off_targets to the spacer dataframe
df = (df
 .join(df_hits.groupby('query_spacer').size().rename('n_off_targets'), on='query_spacer', how='left')
 .fillna({'n_off_targets': 0})
 .astype({'n_off_targets': 'uint32'})
)

# Get information of the off-targets loci. Rank them according to that information and select the worst one per query spacer as a reference to rank each
df_hits_gene_worst = (bioframe.overlap(df1 = df_hits,
                  df2 = df_genes.assign(chrom = '0').loc[:, ['locus_tag', 'gene', 'matches_up_strand', 'start', 'end', 'chrom']],
                  suffixes=('', '_off_target'),
                  how = 'left')
        .drop(columns = ['chrom_off_target', 'chrom'])
        # .astype({'matches_up_strand_gene': 'boolean'})
        # Binds non-template (coding) strand is equivalent to the spacer sequence matching the template (non-coding) strand of the gene
        .assign(binds_non_template_strand_off_target = lambda df_: df_['matches_up_strand'] != df_['matches_up_strand_off_target'])
        # .loc[lambda df_: df_['query_end_position_seed'] != df_['seed_end_seed']] # Filter out the on-targets
        .sort_values(by = ['locus_tag', 'query_spacer', 'binds_non_template_strand_off_target', 'n_nts_mismatched'],
                     ascending = [True, True, False, True])
        # Remove query_spacers whose full_sgrna has any off target to a gene in genes_to_remove (biofilm-inhibiting genes)
        # .loc[lambda df_:~df_['full_sgrna'].isin(
        #     df_.loc[df_['gene_off_target'].isin(genes_to_remove), 'full_sgrna'].unique())]
        .reset_index(drop=True)
)

# Get full_sgrnas to remove
spacers_to_remove = df_hits_gene_worst.loc[df_hits_gene_worst['gene_off_target'].isin(genes_to_remove), 'full_sgrna'].unique()

df_hits_gene_worst = (df_hits_gene_worst
                  .groupby('query_spacer')
                  .head(1)
)


In [ ]:

# Merge this information back to the query spacer dataframe to have it ready for ranking and binning
cols_first = ['locus_tag', 'gene', 'bin_index', 'n_off_targets', 'n_nts_mismatched', 'binds_non_template_strand_off_target']
df = (df
      
        # Remove spacers that will off-target to biofilm-reppressing genes
        .loc[lambda df_:~df_['full_sgrna'].isin(spacers_to_remove)]
        
        .merge(df_hits_gene_worst.drop(columns = set(df.columns) & set(df_hits_gene_worst.columns) - set(['query_spacer'])), on='query_spacer', how='left')
        .reset_index(drop=True)
        .loc[:, lambda df_: cols_first + [col for col in df_.columns if col not in cols_first]]
        .assign(selected = False)
        # Do a hard filter: Include only guides with less than 10 off-targets
        .loc[lambda df_: df_['n_off_targets'] <= 10]
        # .columns
)

In [ ]:
# Perform ranking and binning based on sort_by and sort_ascending
sort_by = ['n_off_targets', 'binds_non_template_strand_off_target', 'n_nts_mismatched',
           'gc_downstream_8_9',
                     'agg_pam', 'n_t_in_start', 'offset_gt_1000']
sort_ascending = [True, True, False,
                  True,
                    True, True, True] # Order of booleans when sorting is ['False', 'True']

# First assign solely based on bin_index_name
df = (df
    # Assign the bin_index_fitness and bin_index (predicted)
    .pipe(generate_spacers.subsample_spacers_per_bin,
          k = 2,
          sort_by = sort_by,
          sort_ascending = sort_ascending, 
          bin_index_name = 'bin_index')

    # Assign the missing bin_index by distance to the closest bin_index
    .groupby(['locus_tag'])
    .apply(generate_spacers.generate_bin_sample_single_gene,
           bin_dict = BIN_DICT,
           k=2,
           sort_by = sort_by,
           sort_ascending = sort_ascending,
           bin_index_name = 'bin_index',
           include_groups = False)
    .reset_index()
    .drop(columns = 'level_1')
    .sort_values(['locus_tag', 'bin_index']+sort_by, ascending=[True, True]+sort_ascending)
)

In [ ]:
df_selected = df.loc[lambda df_:df_['selected']]
df_selected.to_pickle(headpath+'df_selected_auxotrophs.pkl')
df.to_pickle(headpath+'df_auxotrophs.pkl')

# Nonessentials

In [ ]:
df_genes = pd.read_pickle(headpath+'df_genes.pkl')

In [ ]:
# Find
path_to_output = headpath+'sgrnas_nonessentials'
locus_tags = df_genes.loc[df_genes['category_gene']=='nonessential','locus_tag_file'].to_list()

In [ ]:

generate_spacers.run_grna_finding_pipeline(locus_tags,
                                            path_to_script,
                                            path_to_genbank,
                                            path_to_params,
                                            path_log,
                                            path_to_output,
                                            get_mismatches = False) # No mismatches for nonessential genes

In [ ]:
# Load the filtered (fully on-target) and the mismatch dataframes
df = (pd.read_csv(path_to_output+'_filter.csv',
                  usecols=['locus_tag', 'gene_name', 'strand', 'n', 'offset',
                           'position_offset', 'gc', 'sgrna', 'pam',
                           'downstream', 'sequence', 'seed', 'off_target_n'])
      .rename(columns={'sgrna': 'full_sgrna', 'gene_name': 'gene'})
      .astype({'strand': 'category',
               'n': 'int32',
               'offset': 'int32',
               'position_offset': 'int32',
               'pam': 'category',
               'off_target_n': 'int32'})
      .assign(predicted_efficacy = 1,
              index_mismatch = 0,
              mismatch_sgrna = lambda df_: df_['full_sgrna'])
      .assign(bin_index = lambda df_:generate_spacers.get_bin_ids(df_,
                                                                    col='predicted_efficacy',
                                                                    bin_dict=BIN_DICT)) # Assign bin indices to each gRNA
      .astype({'predicted_efficacy': 'float64', 'bin_index': 'int8'})

      .loc[lambda df_: df_['full_sgrna'].duplicated(keep=False) == False] # Remove duplicate sgRNAs (both of them)    
      .loc[lambda df_: df_['full_sgrna'].apply(generate_spacers.check_all_sites) == False]
      .loc[lambda df_: df_['full_sgrna'].apply(generate_spacers.add_adapter_seqs).apply(generate_spacers.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Remove rows with BsaI and PvuI sites    
      .sort_values(by=['locus_tag', 'n', 'index_mismatch'])
      .reset_index(drop=True)

      .assign(query_spacer = lambda df_: df_['full_sgrna'])
#       .rename(columns={'full_sgrna': 'query_spacer'})
      .assign(query_end_position = lambda df_: df_['position_offset'])

      # CORRECT LOCUS_TAGS AND GENE NAMES
        .rename(columns={'locus_tag': 'locus_tag_file'})
        .drop(columns = ['gene'])
        .merge(df_genes.loc[:,['locus_tag_file', 'locus_tag', 'gene']],
                left_on='locus_tag_file',
                right_on='locus_tag_file',
                how='left')
        .drop(columns= ['locus_tag_file'])
        .loc[:, lambda df_: ['locus_tag', 'gene']+ [col for col in df_.columns if col not in ['locus_tag', 'gene']]]
        .sort_values(by=['locus_tag', 'n', 'index_mismatch'])

      .assign(gc_downstream_8_9 = lambda df_: df_['downstream'].str[7:9].str.count('G') + df_['downstream'].str[7:9].str.count('C'),
              offset_gt_1000 = lambda df_: df_['offset'] > 1000,
              # Assign a column that counts the number of Ts in the first 3 bases of the query_spacer
              n_t_in_start = lambda df_: df_['query_spacer'].str[:3].str.count('T'),
              agg_pam = lambda df_: df_['pam'].str.contains('AGG'))                                             
)

In [ ]:
# Get unique seed sequences
df_seeds = (df
            .loc[:,['seed']]
            .drop_duplicates()
            .reset_index(drop=True)
)

# df_seeds_all = generate_spacers.find_all_hits_seeds_parallel(df_seeds, upper_seq, lower_seq)
# df_seeds_all = generate_spacers.find_all_hits_seeds(df_seeds, upper_seq, lower_seq)
df_seeds_all.to_pickle(headpath+'df_seeds_all_nonessential.pkl')
# df_seeds_all = pd.read_pickle(headpath+'df_seeds_all_nonessential.pkl')

In [ ]:
df_seeds_all

In [ ]:
# Get information about the off_targets per each query_spacer
df_hits = (df
           .merge(df_seeds_all, on='seed', how='left')
           .assign(n_nts_mismatched = lambda df_: df_.apply(lambda row: generate_spacers.count_differences(row['full_sgrna'], row['spacer_seq']), axis=1))
           .loc[lambda df_: ~df_['n_nts_mismatched'].isna()] # Keep only off-targets (remove the ones with mismatches on the seed)
           .loc[lambda df_: df_['n_nts_mismatched'] > 0] # Keep only off-targets
           .astype({"start": 'int64', "end": 'int64'})
) 

# Add the number of off_targets to the spacer dataframe
df = (df
 .join(df_hits.groupby('query_spacer').size().rename('n_off_targets'), on='query_spacer', how='left')
 .fillna({'n_off_targets': 0})
 .astype({'n_off_targets': 'uint32'})
)


In [ ]:

# Get information of the off-targets loci. Rank them according to that information and select the worst one per query spacer as a reference to rank each
df_hits_gene_worst = (bioframe.overlap(df1 = df_hits,
                  df2 = df_genes.assign(chrom = '0').loc[:, ['locus_tag', 'gene', 'matches_up_strand', 'start', 'end', 'chrom']],
                  suffixes=('', '_off_target'),
                  how = 'left')
        .drop(columns = ['chrom_off_target', 'chrom'])
        # .astype({'matches_up_strand_gene': 'boolean'})
        # Binds non-template (coding) strand is equivalent to the spacer sequence matching the template (non-coding) strand of the gene
        .assign(binds_non_template_strand_off_target = lambda df_: df_['matches_up_strand'] != df_['matches_up_strand_off_target'])
        # .loc[lambda df_: df_['query_end_position_seed'] != df_['seed_end_seed']] # Filter out the on-targets
        .sort_values(by = ['locus_tag', 'query_spacer', 'binds_non_template_strand_off_target', 'n_nts_mismatched'],
                     ascending = [True, True, False, True])
        
        .reset_index(drop=True)
)

# Get full_sgrnas to remove
spacers_to_remove = df_hits_gene_worst.loc[df_hits_gene_worst['gene_off_target'].isin(genes_to_remove), 'full_sgrna'].unique()

df_hits_gene_worst = (df_hits_gene_worst
                  .groupby('query_spacer')
                  .head(1)
)

In [ ]:

# Merge this information back to the query spacer dataframe to have it ready for ranking and binning
cols_first = ['locus_tag', 'gene', 'bin_index', 'n_off_targets', 'n_nts_mismatched', 'binds_non_template_strand_off_target']
df = (df
        # Remove spacers that will off-target to biofilm-reppressing genes
        .loc[lambda df_:~df_['full_sgrna'].isin(spacers_to_remove)]
        
        .merge(df_hits_gene_worst.drop(columns = set(df.columns) & set(df_hits_gene_worst.columns) - set(['query_spacer'])), on='query_spacer', how='left')
        .reset_index(drop=True)
        .loc[:, lambda df_: cols_first + [col for col in df_.columns if col not in cols_first]]
        .assign(selected = False)
        # Do a hard filter: Include only guides with less than 10 off-targets
        .loc[lambda df_: df_['n_off_targets'] <= 10]
        # .columns
)

In [ ]:
# Perform ranking and binning based on sort_by and sort_ascending
sort_by = ['n_off_targets', 'binds_non_template_strand_off_target', 'n_nts_mismatched',
           'gc_downstream_8_9',
                     'agg_pam', 'n_t_in_start', 'offset_gt_1000']
sort_ascending = [True, True, False,
                  True,
                    True, True, True] # Order of booleans when sorting is ['False', 'True']

# First assign solely based on bin_index_name
df = (df
    # Assign the bin_index_fitness and bin_index (predicted)
    .pipe(generate_spacers.subsample_spacers_per_bin,
          k = 3,
          sort_by = sort_by,
          sort_ascending = sort_ascending, 
          bin_index_name = 'bin_index')

    # # Assign the missing bin_index by distance to the closest bin_index
    # .groupby(['locus_tag'])
    # .apply(generate_spacers.generate_bin_sample_single_gene,
    #        bin_dict = BIN_DICT,
    #        k=2,
    #        sort_by = sort_by,
    #        sort_ascending = sort_ascending,
    #        bin_index_name = 'bin_index',
    #        include_groups = False)
    .reset_index()
    # .drop(columns = 'level_1')
    .sort_values(['locus_tag', 'bin_index']+sort_by, ascending=[True, True]+sort_ascending)
)

In [ ]:
df_selected = df.loc[lambda df_:df_['selected']]
df_selected.to_pickle(headpath+'df_selected_nonessential.pkl')
df.to_pickle(headpath+'df_nonessential.pkl')

In [ ]:
df['n_off_targets'].hist(density=True, alpha=0.5, label='All')
df_selected['off_target_n'].hist(density=True, alpha=0.5, label='Selected')

# tRNAs

In [ ]:
df_genes = pd.read_pickle(headpath+'df_genes.pkl')
df_genes_trnas = df_genes.loc[df_genes['category_gene']=='t_rna']

In [ ]:
path_to_output = headpath+'sgrnas_trnas'
path_log = headpath+'log.txt'

BIN_DICT = {
    0:[-0.5, 0.1], # Lower bound 
    1:[0.1, 0.2],
    2:[0.2, 0.3],
    3:[0.3, 0.4],
    4:[0.4, 0.5],
    5:[0.5, 0.6],
    6:[0.6, 0.7],
    7:[0.7, 0.8],
    8:[0.8, 0.9],
    9:[0.9, 1.5] # Cannot set upper bound bc there are fitness values >= 1
}

In [ ]:
locus_tags_trnas = df_genes.loc[df_genes['category_gene']=='t_rna','locus_tag_file'].to_list()

generate_spacers.run_grna_finding_pipeline(locus_tags_trnas,
                                            path_to_script,
                                            path_to_genbank,
                                            path_to_params,
                                            path_log,
                                            path_to_output,
                                            get_mismatches = True)

In [ ]:
df = generate_spacers.load_and_process_dfs_from_pipeline(path_to_output, df_genes, BIN_DICT)

In [ ]:
# Get unique seed sequences
df_seeds = (df
            .loc[:,['seed']]
            .drop_duplicates()
            .reset_index(drop=True)
)

df_seeds_all = generate_spacers.find_all_hits_seeds_parallel(df_seeds, upper_seq, lower_seq)
# df_seeds_all = generate_spacers.find_all_hits_seeds(df_seeds, upper_seq, lower_seq)
df_seeds_all.to_pickle(headpath+'df_seeds_all_trna.pkl')
# df_seeds_all = pd.read_pickle(headpath+'df_seeds_all_trna.pkl')

In [ ]:
# Changed pipeline from the one used for auxotrophs and nonessentials. Here I cannot assume the ones with zero off targets
category_identifier = 'tRNA'
df_hits = (df
           .merge(df_seeds_all, on='seed', how='left')
           .assign(n_nts_mismatched = lambda df_: df_.apply(lambda row: generate_spacers.count_differences(row['full_sgrna'], row['spacer_seq']), axis=1))
           .loc[lambda df_: ~df_['n_nts_mismatched'].isna()] # Keep only off-targets (remove the ones with mismatches on the seed, so they don't even target themselves)

           # I CANNOT USE THIS CRITERION TO KEEP ONLY OFF-TARGETS IN THIS CASE. I WILL FILTER BASED ON THE IDENTITY OF THE OFF_TARGET
        #    .loc[lambda df_: df_['n_nts_mismatched'] > 0] # Keep only off-targets
           .astype({"start": 'int64', "end": 'int64'})
)

# Get information of the off-targets loci. Rank them according to that information. DO NOT ONLY KEEP THE FIRST ONE IN THIS CASE
df_hits_gene_worst = (bioframe.overlap(df1 = df_hits,
                  df2 = df_genes.assign(chrom = '0').loc[:, ['locus_tag', 'gene', 'matches_up_strand', 'start', 'end', 'chrom']],
                  suffixes=('', '_off_target'),
                  how = 'left')
        .drop(columns = ['chrom_off_target', 'chrom'])
        # .astype({'matches_up_strand_gene': 'boolean'})
        # Binds non-template (coding) strand is equivalent to the spacer sequence matching the template (non-coding) strand of the gene
        .assign(binds_non_template_strand_off_target = lambda df_: df_['matches_up_strand'] != df_['matches_up_strand_off_target'])
        # .loc[lambda df_: df_['query_end_position_seed'] != df_['seed_end_seed']] # Filter out the on-targets
        .sort_values(by = ['locus_tag', 'query_spacer', 'binds_non_template_strand_off_target', 'n_nts_mismatched'],
                     ascending = [True, True, False, True])
        # .sort_values(by = ['query_index_seed', 'binds_non_template_strand', 'n_nts_mismatched_seed'], ascending = [True, True, False])
        .reset_index(drop=True)
        # DO NOT ONLY KEEP THE FIRST ONE IN THIS CASE
        # .groupby('query_spacer')
        # .head(1)
)

# Get full_sgrnas to remove
spacers_to_remove = df_hits_gene_worst.loc[df_hits_gene_worst['gene_off_target'].isin(genes_to_remove), 'full_sgrna'].unique()

# REMOVE ON-TARGETS (VERIFY THAT THERE IS EXACTLY ONE IF COMPARING BY GENE NAME)
assert len(df) == len(df_hits_gene_worst.loc[df_hits_gene_worst['gene'] == df_hits_gene_worst['gene_off_target']])
df_hits_gene_worst = (df_hits_gene_worst
                      .loc[lambda df_:~(df_['gene'] == df_['gene_off_target'])]
)

# Add the number of off_targets to the spacer dataframe
df = (df
 .join(df_hits_gene_worst.groupby('query_spacer').size().rename('n_off_targets'), on='query_spacer', how='left')
 .fillna({'n_off_targets': 0})
 .astype({'n_off_targets': 'uint32'})
)

# Count the number of off-targets to another tRNA, and to any other gene
df = (df
    #   # n_off_target_locus
    #   .join(df_hits_gene_worst
    #         .groupby('query_spacer')
    #         .apply(lambda x: (x['locus_tag'] == x['locus_tag_off_target']).sum(),
    #                include_groups=False)
    #         .rename('n_off_target_locus'),
    #        on='query_spacer',
    #        how='left')
     # n_off_target_tRNAs
      .join(df_hits_gene_worst
            .groupby('query_spacer')
            .apply(lambda x: (x['locus_tag_off_target'].str.contains(category_identifier)).sum(),
                   include_groups=False)
            .rename('n_off_target_'+ category_identifier),
           on='query_spacer',
           how='left')
     # n_off_target_out
      .join(df_hits_gene_worst
            .groupby('query_spacer')
            .apply(lambda x: (~x['locus_tag_off_target'].fillna('').str.contains(category_identifier)).sum(),
                   include_groups=False)
            .rename('n_off_target_other'),
           on='query_spacer',
           how='left')
)

# Now get the worst one from df_hits_gene_worst and continue with the pipeline as usual
df_hits_gene_worst = (df_hits_gene_worst
 .loc[lambda df_:~df_['locus_tag_off_target'].fillna('').str.contains(category_identifier)]
 .sort_values(by = ['locus_tag', 'query_spacer', 'binds_non_template_strand_off_target', 'n_nts_mismatched'],
                     ascending = [True, True, False, True])
 .reset_index(drop=True)
 .groupby('query_spacer')
 .head(1)
)

# Merge this information back to the query spacer dataframe to have it ready for ranking and binning
cols_first = ['locus_tag', 'gene', 'bin_index', 'n_off_targets', 'n_nts_mismatched', 'binds_non_template_strand_off_target']
df = (df
      
       # Remove spacers that will off-target to biofilm-reppressing genes
        .loc[lambda df_:~df_['full_sgrna'].isin(spacers_to_remove)]
        
        .merge(df_hits_gene_worst.drop(columns = set(df.columns) & set(df_hits_gene_worst.columns) - set(['query_spacer'])), on='query_spacer', how='left')
        .reset_index(drop=True)
        .loc[:, lambda df_: cols_first + [col for col in df_.columns if col not in cols_first]]
        .assign(selected = False)
        # Do a hard filter: Include only guides with less than 10 off-targets
        .loc[lambda df_: df_['n_off_targets'] <= 10]
)

In [ ]:
# Perform ranking and binning based on sort_by and sort_ascending
sort_by = ['n_off_targets', 'n_off_target_other', 'n_off_target_'+ category_identifier,
           'binds_non_template_strand_off_target', 'n_nts_mismatched',
           'gc_downstream_8_9',
                     'agg_pam', 'n_t_in_start', 'offset_gt_1000']
sort_ascending = [True, True, True,
                  True, False,
                  True,
                    True, True, True] # Order of booleans when sorting is ['False', 'True']

df = (df
    # Assign the bin_index_fitness and bin_index (predicted)
    # .pipe(subsample_spacers_per_bin, k = 1, col = 'relative_fitness_mean', bin_index_name = 'bin_index_fitness')
    .pipe(generate_spacers.subsample_spacers_per_bin,
          k = 1,
          sort_by = sort_by,
          sort_ascending = sort_ascending, 
          bin_index_name = 'bin_index')

    # Assign the missing bin_index by distance to the closest bin_index
    .groupby(['locus_tag'])
    .apply(generate_spacers.generate_bin_sample_single_gene,
           bin_dict = BIN_DICT,
           k=1,
           sort_by = sort_by,
           sort_ascending = sort_ascending,
           bin_index_name = 'bin_index',
           include_groups = False)
    .reset_index()
    .drop(columns = 'level_1')

    .sort_values(['locus_tag', 'bin_index']+sort_by, ascending=[True, True]+sort_ascending)
)

In [ ]:
df_selected = df.loc[lambda df_:df_['selected']]
df_selected.to_pickle(headpath+'df_selected_trna.pkl')
df.to_pickle(headpath+'df_trna.pkl')

# rRNAs

In [ ]:
df_genes = pd.read_pickle(headpath+'df_genes.pkl')
df_genes_rrnas = df_genes.loc[df_genes['category_gene']=='r_rna']

In [ ]:
path_to_output = headpath+'sgrnas_rrnas'

BIN_DICT = {
    0:[-0.5, 0.1], # Lower bound 
    1:[0.1, 0.2],
    2:[0.2, 0.3],
    3:[0.3, 0.4],
    4:[0.4, 0.5],
    5:[0.5, 0.6],
    6:[0.6, 0.7],
    7:[0.7, 0.8],
    8:[0.8, 0.9],
    9:[0.9, 1.5] # Cannot set upper bound bc there are fitness values >= 1
}

In [ ]:
locus_tags_rrnas = df_genes.loc[df_genes['category_gene']=='r_rna','locus_tag_file'].to_list()

generate_spacers.run_grna_finding_pipeline(locus_tags_rrnas,
                                            path_to_script,
                                            path_to_genbank,
                                            path_to_params,
                                            path_log,
                                            path_to_output,
                                            get_mismatches = True)

In [ ]:
df = generate_spacers.load_and_process_dfs_from_pipeline(path_to_output, df_genes, BIN_DICT)

In [ ]:
# Get unique seed sequences
df_seeds = (df
            .loc[:,['seed']]
            .drop_duplicates()
            .reset_index(drop=True)
)

df_seeds_all = generate_spacers.find_all_hits_seeds_parallel(df_seeds, upper_seq, lower_seq)
# df_seeds_all = generate_spacers.find_all_hits_seeds(df_seeds, upper_seq, lower_seq)
df_seeds_all.to_pickle(headpath+'df_seeds_all_rrna.pkl')
# df_seeds_all = pd.read_pickle(headpath+'df_seeds_all_rrna.pkl')

In [ ]:
# Changed pipeline from the one used for auxotrophs and nonessentials. Here I cannot assume the ones with zero off targets
category_identifier = 'rRNA'
df_hits = (df
           .merge(df_seeds_all, on='seed', how='left')
           .assign(n_nts_mismatched = lambda df_: df_.apply(lambda row: generate_spacers.count_differences(row['full_sgrna'], row['spacer_seq']), axis=1))
           .loc[lambda df_: ~df_['n_nts_mismatched'].isna()] # Keep only off-targets (remove the ones with mismatches on the seed, so they don't even target themselves)

           # I CANNOT USE THIS CRITERION TO KEEP ONLY OFF-TARGETS IN THIS CASE. I WILL FILTER BASED ON THE IDENTITY OF THE OFF_TARGET
        #    .loc[lambda df_: df_['n_nts_mismatched'] > 0] # Keep only off-targets
           .astype({"start": 'int64', "end": 'int64'})
)

# Get information of the off-targets loci. Rank them according to that information. DO NOT ONLY KEEP THE FIRST ONE IN THIS CASE
df_hits_gene_worst = (bioframe.overlap(df1 = df_hits,
                  df2 = df_genes.assign(chrom = '0').loc[:, ['locus_tag', 'gene', 'matches_up_strand', 'start', 'end', 'chrom']],
                  suffixes=('', '_off_target'),
                  how = 'left')
        .drop(columns = ['chrom_off_target', 'chrom'])
        # .astype({'matches_up_strand_gene': 'boolean'})
        # Binds non-template (coding) strand is equivalent to the spacer sequence matching the template (non-coding) strand of the gene
        .assign(binds_non_template_strand_off_target = lambda df_: df_['matches_up_strand'] != df_['matches_up_strand_off_target'])
        # .loc[lambda df_: df_['query_end_position_seed'] != df_['seed_end_seed']] # Filter out the on-targets
        .sort_values(by = ['locus_tag', 'query_spacer', 'binds_non_template_strand_off_target', 'n_nts_mismatched'],
                     ascending = [True, True, False, True])
        # .sort_values(by = ['query_index_seed', 'binds_non_template_strand', 'n_nts_mismatched_seed'], ascending = [True, True, False])
        .reset_index(drop=True)
        # DO NOT ONLY KEEP THE FIRST ONE IN THIS CASE
        # .groupby('query_spacer')
        # .head(1)
)

# REMOVE ON-TARGETS (VERIFY THAT THERE IS EXACTLY ONE IF COMPARING BY GENE NAME)
assert len(df) == len(df_hits_gene_worst.loc[df_hits_gene_worst['gene'] == df_hits_gene_worst['gene_off_target']])
df_hits_gene_worst = (df_hits_gene_worst
                      .loc[lambda df_:~(df_['gene'] == df_['gene_off_target'])]
)

# Get full_sgrnas to remove
spacers_to_remove = df_hits_gene_worst.loc[df_hits_gene_worst['gene_off_target'].isin(genes_to_remove), 'full_sgrna'].unique()

# Add the number of off_targets to the spacer dataframe
df = (df
 .join(df_hits_gene_worst.groupby('query_spacer').size().rename('n_off_targets'), on='query_spacer', how='left')
 .fillna({'n_off_targets': 0})
 .astype({'n_off_targets': 'uint32'})
)

# Count the number of off-targets to another tRNA, and to any other gene
df = (df
    #   # n_off_target_locus
    #   .join(df_hits_gene_worst
    #         .groupby('query_spacer')
    #         .apply(lambda x: (x['locus_tag'] == x['locus_tag_off_target']).sum(),
    #                include_groups=False)
    #         .rename('n_off_target_locus'),
    #        on='query_spacer',
    #        how='left')
     # n_off_target_tRNAs
      .join(df_hits_gene_worst
            .groupby('query_spacer')
            .apply(lambda x: (x['locus_tag_off_target'].str.contains(category_identifier)).sum(),
                   include_groups=False)
            .rename('n_off_target_'+ category_identifier),
           on='query_spacer',
           how='left')
     # n_off_target_out
      .join(df_hits_gene_worst
            .groupby('query_spacer')
            .apply(lambda x: (~x['locus_tag_off_target'].fillna('').str.contains(category_identifier)).sum(),
                   include_groups=False)
            .rename('n_off_target_other'),
           on='query_spacer',
           how='left')
)

# Now get the worst one from df_hits_gene_worst and continue with the pipeline as usual
df_hits_gene_worst = (df_hits_gene_worst
 .loc[lambda df_:~df_['locus_tag_off_target'].fillna('').str.contains(category_identifier)]
 .sort_values(by = ['locus_tag', 'query_spacer', 'binds_non_template_strand_off_target', 'n_nts_mismatched'],
                     ascending = [True, True, False, True])
 .reset_index(drop=True)
 .groupby('query_spacer')
 .head(1)
)

# Merge this information back to the query spacer dataframe to have it ready for ranking and binning
cols_first = ['locus_tag', 'gene', 'bin_index', 'n_off_targets', 'n_nts_mismatched', 'binds_non_template_strand_off_target']
df = (df
        # Remove spacers that will off-target to biofilm-reppressing genes
        .loc[lambda df_:~df_['full_sgrna'].isin(spacers_to_remove)]

        .merge(df_hits_gene_worst.drop(columns = set(df.columns) & set(df_hits_gene_worst.columns) - set(['query_spacer'])), on='query_spacer', how='left')
        .reset_index(drop=True)
        .loc[:, lambda df_: cols_first + [col for col in df_.columns if col not in cols_first]]
        .assign(selected = False)
        # Do a hard filter: Include only guides with less than 10 off-targets
        .loc[lambda df_: df_['n_off_targets'] <= 10]
)

In [ ]:
# Perform ranking and binning based on sort_by and sort_ascending
sort_by = ['n_off_targets', 'n_off_target_other', 'n_off_target_'+ category_identifier,
           'binds_non_template_strand_off_target', 'n_nts_mismatched',
           'gc_downstream_8_9',
                     'agg_pam', 'n_t_in_start', 'offset_gt_1000']
sort_ascending = [True, True, True,
                  True, False,
                  True,
                    True, True, True] # Order of booleans when sorting is ['False', 'True']

df = (df
    # Assign the bin_index_fitness and bin_index (predicted)
    # .pipe(subsample_spacers_per_bin, k = 1, col = 'relative_fitness_mean', bin_index_name = 'bin_index_fitness')
    .pipe(generate_spacers.subsample_spacers_per_bin,
          k = 2,
          sort_by = sort_by,
          sort_ascending = sort_ascending, 
          bin_index_name = 'bin_index')

    # Assign the missing bin_index by distance to the closest bin_index
    .groupby(['locus_tag'])
    .apply(generate_spacers.generate_bin_sample_single_gene,
           bin_dict = BIN_DICT,
           k=2,
           sort_by = sort_by,
           sort_ascending = sort_ascending,
           bin_index_name = 'bin_index',
           include_groups = False)
    .reset_index()
    .drop(columns = 'level_1')

    .sort_values(['locus_tag', 'bin_index']+sort_by, ascending=[True, True]+sort_ascending)
)

In [ ]:
df_selected = df.loc[lambda df_:df_['selected']]
df_selected.to_pickle(headpath+'df_selected_rrna.pkl')
df.to_pickle(headpath+'df_rrna.pkl')

# Consolidate all data

In [ ]:
df_ess_sel = (pd.read_pickle(headpath+'df_selected_essentials_from_dataset.pkl')
              .assign(gene_category = 'essential')
)

df_ctr_sel = (pd.read_pickle(headpath+'df_selected_controls_from_dataset.pkl')
              .assign(gene_category = 'control')
)

df_aux_sel = (pd.read_pickle(headpath+'df_selected_auxotrophs.pkl')
              .assign(gene_category = 'auxotroph')
)

df_non_sel = (pd.read_pickle(headpath+'df_selected_nonessential.pkl')
              .assign(gene_category = 'nonessential')
)

df_trna_sel = (pd.read_pickle(headpath+'df_selected_trna.pkl')
                .assign(gene_category = 'tRNA')
)

df_rrna_sel = (pd.read_pickle(headpath+'df_selected_rrna.pkl')
                .assign(gene_category = 'rRNA')
)

# df_bfl_sel = (pd.read_pickle(headpath+'df_selected_biofilm.pkl')
#               .assign(gene_category = 'nonessential')
# )

In [ ]:
df_ess_all = pd.concat([df_ess_sel, df_aux_sel, df_trna_sel, df_rrna_sel, df_ctr_sel], axis=0)

In [ ]:
df_non_all = pd.concat([df_non_sel, df_ctr_sel], axis=0)

In [ ]:
def generate_seqs_to_order(df, genes_to_remove, suffix_to_remove, suffix_other):
    df = (df
        #   .loc[lambda df_:~df_['gene'].isin(genes_to_remove)]
          .assign(seq_to_order = lambda df_: df_['query_spacer'].apply(generate_spacers.add_adapter_seqs))
          .assign(seq_to_order_with_suffixes = lambda df_: np.where(df_['gene'].isin(genes_to_remove),
                                                       df_['seq_to_order'] + suffix_to_remove,
                                                       df_['seq_to_order'] + suffix_other))
                                            )
    assert (~df['seq_to_order_with_suffixes'].apply(generate_spacers.check_all_sites,  args=[["GGTCTC", "CGATCG"]])).all()
    assert (~df['seq_to_order'].apply(generate_spacers.check_all_sites,  args=[["GGTCTC", "CGATCG"]])).all()
    print("No BsaI or PvuI restriction sites")
    return df

In [ ]:
# def generate_seqs_to_order(df):
#     df = (df
#           .assign(seq_to_order = lambda df_: df_['query_spacer'].apply(generate_spacers.add_adapter_seqs))
#     )
#     assert (~df['seq_to_order'].apply(generate_spacers.check_all_sites,  args=[["GGTCTC", "CGATCG"]])).all()
#     print("No BsaI or PvuI restriction sites")
#     return df

In [ ]:
df_ess_to_order_all = generate_seqs_to_order(df_ess_all, genes_to_remove, suffix_to_remove, suffix_other)
df_non_to_order_all = generate_seqs_to_order(df_non_all, genes_to_remove, suffix_to_remove, suffix_other)

# df_non
df_non_to_order_all.to_csv(headpath+'df_non_to_order_all.csv', index=False)
df_non_to_order_all.to_pickle(headpath+'df_non_to_order_all.pkl')

# Same for df_ess_to_order_all
df_ess_to_order_all.to_csv(headpath+'df_ess_to_order_all.csv', index=False)
df_ess_to_order_all.to_pickle(headpath+'df_ess_to_order_all.pkl')

In [ ]:
# df_ess_to_order_simplified
df_ess_to_order_simplified = (df_ess_to_order_all
                              .assign(name = 'opLAG_1')
                              .loc[:,['name', 'seq_to_order_with_suffixes']]
)

# Same for df_non
df_non_to_order_simplified = (df_non_to_order_all
                              .assign(name = 'opLAG_2')
                              .loc[:,['name', 'seq_to_order_with_suffixes']]
)

In [ ]:
df_ess_to_order_simplified.to_csv(headpath+'df_ess_to_order_simplified.csv', index=False)
df_non_to_order_simplified.to_csv(headpath+'df_non_to_order_simplified.csv', index=False)

In [ ]:
df_ess_to_order_simplified= pd.read_csv(headpath+'df_ess_to_order_simplified.csv')
df_non_to_order_simplified= pd.read_csv(headpath+'df_non_to_order_simplified.csv')

In [4]:
df_ess_to_order_all= pd.read_pickle(headpath+'df_ess_to_order_all.pkl')
df_non_to_order_all= pd.read_pickle(headpath+'df_non_to_order_all.pkl')

In [6]:
df_ess_to_order_all['gene'].nunique()

410